## Importation des bibliothèques

In [21]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Racine du projet = .../Fil-rouge 
PROJECT_ROOT = Path.cwd().parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

PROJECT_ROOT: c:\Users\imane\Desktop\Fil-rouge
RAW_DIR: c:\Users\imane\Desktop\Fil-rouge\data\raw
PROCESSED_DIR: c:\Users\imane\Desktop\Fil-rouge\data\processed


## Chargement des datasets

In [28]:
df_fastf1 = pd.read_csv(RAW_DIR / "fastf1_kpis_R_Q_2021_2025.csv")
df_constructor_fin = pd.read_csv(RAW_DIR / "f1_constructor_finance_2021_2025.csv")
df_driver = pd.read_csv(RAW_DIR / "f1_driver_finance_2021_2025.csv")


## 1. Nettoyage Driver Finance

In [ ]:

# ==============================
# CONFIG - ARCHITECTURE PROJET
# ==============================
PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "f1_driver_finance_2021_2025.csv"
OUTPUT_PATH = PROCESSED_DIR / "driver_finance_clean.csv"

# ==============================
# PARSE SALARY
# ==============================
def parse_salary_to_usd(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return np.nan

    s = str(value).strip().lower()
    if s in {"", "nan", "none"}:
        return np.nan

    s = s.replace(",", ".")
    s = s.replace("million", "m").replace("$", "").replace("€", "").replace("£", "")
    s = re.sub(r"[^0-9\.\-\sm]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # intervalle ex: 10-12m
    m = re.match(r"^\s*(\d+(\.\d+)?)\s*-\s*(\d+(\.\d+)?)\s*m?\s*$", s)
    if m:
        a = float(m.group(1))
        b = float(m.group(3))
        return ((a + b) / 2.0) * 1_000_000

    # valeur simple
    m = re.search(r"(\d+(\.\d+)?)", s)
    if m:
        a = float(m.group(1))
        if "m" in s:
            return a * 1_000_000
        return a

    return np.nan

# ==============================
# CLEAN FUNCTION
# ==============================
def clean_driver_finance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # -------- 1) Nettoyage texte
    text_cols = ["driverId", "driverName", "constructorId", "salary_raw", "salarySourceURL"]
    for c in text_cols:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
            df[c] = df[c].replace({"nan": "", "None": "", "": np.nan})

    # Harmonisation IDs
    if "driverId" in df.columns:
        df["driverId"] = df["driverId"].astype(str).str.lower().str.strip()
    if "constructorId" in df.columns:
        df["constructorId"] = df["constructorId"].astype(str).str.lower().str.strip()

    # -------- 2) Types
    df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    df["driverRank"] = pd.to_numeric(df["driverRank"], errors="coerce").astype("Int64")
    df["driverPoints"] = pd.to_numeric(df["driverPoints"], errors="coerce")
    df["estimatedSalaryUSD"] = pd.to_numeric(df["estimatedSalaryUSD"], errors="coerce")

    # -------- 3) Reconstruction depuis salary_raw
    if "salary_raw" in df.columns:
        parsed_salary = df["salary_raw"].apply(parse_salary_to_usd)
        df["estimatedSalaryUSD"] = df["estimatedSalaryUSD"].fillna(parsed_salary)

    # -------- 4) Contrôles salaire
    df.loc[df["estimatedSalaryUSD"] < 0, "estimatedSalaryUSD"] = np.nan

    # -------- 5) Indicateurs qualité
    df["salaryMissingFlag"] = df["estimatedSalaryUSD"].isna().astype(int)
    df["salaryCoverageStatus"] = np.where(
        df["estimatedSalaryUSD"].isna(),
        "missing_public_salary",
        "salary_available"
    )

    # -------- 6) Option d’imputation légère (séparée)
    # On garde la vraie colonne telle quelle
    # et on crée une colonne imputée pour certains usages analytiques
    df["estimatedSalaryUSD_imputed"] = df["estimatedSalaryUSD"]

    # imputation par médiane des pilotes de la même saison
    df["estimatedSalaryUSD_imputed"] = df.groupby("season")["estimatedSalaryUSD_imputed"]\
        .transform(lambda s: s.fillna(s.median()))

    # -------- 7) Suppression lignes critiques
    df = df.dropna(subset=["season", "driverId"])

    # -------- 8) Doublons métier
    df = df.drop_duplicates(subset=["season", "driverId"], keep="first")

    # -------- 9) Reset index
    df.reset_index(drop=True, inplace=True)

    return df

# ==============================
# LOAD -> CLEAN -> EXPORT
# ==============================
df_driver = pd.read_csv(INPUT_PATH)
df_driver_clean = clean_driver_finance(df_driver)

df_driver_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print(" Dataset driver finance nettoyé et exporté")

print(" Shape :", df_driver_clean.shape)

print("\nPilotes par saison :")
print(df_driver_clean.groupby("season")["driverId"].nunique())

print("\nSalaires manquants par saison :")
print(df_driver_clean.groupby("season")["estimatedSalaryUSD"].apply(lambda x: x.isna().sum()))

print("\nCouverture salaire :")
print(df_driver_clean["salaryCoverageStatus"].value_counts())

print("\nExemples lignes avec salaire manquant :")
print(
    df_driver_clean[df_driver_clean["estimatedSalaryUSD"].isna()][
        ["season", "driverId", "driverName", "constructorId", "salaryCoverageStatus"]
    ].head(15)
)

### 1.1. Test après nettoyage

In [ ]:
# ============================================================
# VALIDATION CLEANING - DRIVER FINANCE DATASET
# ============================================================

print("========================================")
print("VALIDATION DRIVER FINANCE CLEANING")
print("========================================\n")

# ============================================================
#  Vérifier shape dataset
# ============================================================

print("Shape dataset:", df_driver_clean.shape)

assert df_driver_clean.shape[0] > 0, " Dataset vide après nettoyage"
assert df_driver_clean.shape[1] >= 10, " Colonnes manquantes"


# ============================================================
#  Vérifier les colonnes importantes
# ============================================================

expected_columns = {
    "season",
    "driverId",
    "driverName",
    "constructorId",
    "driverRank",
    "driverPoints",
    "salary_raw",
    "estimatedSalaryUSD",
    "salaryMissingFlag",
    "salaryCoverageStatus",
    "estimatedSalaryUSD_imputed"
}

missing_cols = expected_columns - set(df_driver_clean.columns)

print("Colonnes manquantes :", missing_cols)

assert len(missing_cols) == 0, " Certaines colonnes attendues sont absentes"


# ============================================================
#  Vérifier doublons métier
# clé métier = season + driverId
# ============================================================

dup = df_driver_clean.duplicated(
    subset=["season","driverId"]
).sum()

print("Doublons métier :", dup)

assert dup == 0, " Des doublons existent"


# ============================================================
#  Vérifier saisons présentes
# ============================================================

seasons = set(df_driver_clean["season"].dropna().unique())

print("Saisons détectées :", seasons)

expected = {2021,2022,2023,2024,2025}

assert seasons == expected, " Saisons incorrectes"


# ============================================================
#  Vérifier driverId non vide
# ============================================================

missing_driver = df_driver_clean["driverId"].isna().sum()

print("driverId manquants :", missing_driver)

assert missing_driver == 0, " driverId manquants"


# ============================================================
#  Vérifier salaires négatifs
# ============================================================

neg_salary = (df_driver_clean["estimatedSalaryUSD"] < 0).sum()

print("Salaires négatifs :", neg_salary)

assert neg_salary == 0, " Salaire négatif détecté"


# ============================================================
#  Vérifier cohérence salaryMissingFlag
# ============================================================

flag_check = df_driver_clean[
    (df_driver_clean["salaryMissingFlag"] == 1) &
    (df_driver_clean["estimatedSalaryUSD"].notna())
]

print("Incohérences salaryMissingFlag :", len(flag_check))

assert len(flag_check) == 0, " salaryMissingFlag incorrect"


# ============================================================
#  Vérifier imputation
# ============================================================

imputed_missing = df_driver_clean["estimatedSalaryUSD_imputed"].isna().sum()

print("Valeurs NaN dans salary imputé :", imputed_missing)

assert imputed_missing == 0, " Imputation incorrecte"


# ============================================================
#  Vérifier distribution salaires
# ============================================================

print("\nDistribution salaires :")

print(
    df_driver_clean["estimatedSalaryUSD"].describe()
)


# ============================================================
#  Vérifier couverture salaires
# ============================================================

print("\nCouverture salaires :")

print(
    df_driver_clean["salaryCoverageStatus"].value_counts()
)


# ============================================================
# TEST FINAL
# ============================================================

print("\n========================================")
print(" DRIVER FINANCE CLEANING VALIDATED")
print("========================================")

## 2. Néttoyage FastF1

In [ ]:

# 1) Filtrer pré-saison
mask_preseason = df_fastf1["gp"].str.contains("Pre-Season", case=False, na=False)
df = df_fastf1.loc[~mask_preseason].copy()

# garder uniquement Q/R
df = df[df["session"].isin(["Q", "R"])].copy()

# 2) Types
df["season"] = df["season"].astype("int64")
df["round"] = df["round"].astype("int64")

for c in ["gp", "session", "circuit", "driverCode"]:
    df[c] = df[c].astype("category")

# 3) Corriger valeurs invalides
df.loc[df["maxSpeed_kmh"] == 0, "maxSpeed_kmh"] = np.nan

for c in ["gridPosition", "finishPosition", "qualyPosition"]:
    if c in df.columns:
        df.loc[df[c] == 0, c] = np.nan

# finishPosition obligatoire en Race
df = df[~((df["session"] == "R") & (df["finishPosition"].isna()))].copy()

# 4) Déduplication
key = ["season", "round", "driverCode", "session"]
df["_na_count"] = df.isna().sum(axis=1)

df = df.sort_values(
    by=key + ["_na_count", "bestLapTime_sec"],
    ascending=[True, True, True, True, True, True]  # key asc + na asc + bestLap asc
)

df = df.drop_duplicates(subset=key, keep="first").drop(columns=["_na_count"])

# 5) startPosition (cohérent Q/R)
df["startPosition"] = np.where(df["session"] == "R", df["gridPosition"], df["qualyPosition"])

# 6) Outliers robustes (winsorisation par circuit+session)
def clip_group(s, low=0.01, high=0.99):
    s = s.astype(float)
    if s.notna().sum() < 10:   # pas assez de points -> on ne clippe pas
        return s
    ql = s.quantile(low)
    qh = s.quantile(high)
    if pd.isna(ql) or pd.isna(qh):
        return s
    return s.clip(lower=ql, upper=qh)

for col in ["bestLapTime_sec", "avgLapTime_sec", "stdLapTime_sec", "maxSpeed_kmh"]:
    df[col] = df.groupby(["circuit", "session"], observed=True)[col].transform(clip_group)

# 7) Imputation légère (ML-ready)
df["maxSpeed_kmh"] = df.groupby(["season", "circuit", "session"], observed=True)["maxSpeed_kmh"]\
                       .transform(lambda s: s.fillna(s.median()))
df["stdLapTime_sec"] = df.groupby(["season", "circuit", "session"], observed=True)["stdLapTime_sec"]\
                         .transform(lambda s: s.fillna(s.median()))

# 8) Race enrichi par Qualy
df_q = df[df["session"] == "Q"][["season", "round", "driverCode", "qualyPosition"]].copy()
df_r = df[df["session"] == "R"].copy()

df_r = df_r.merge(df_q, on=["season", "round", "driverCode"], how="left", suffixes=("", "_fromQ"))

# raceGain (uniquement Race)
df_r["raceGain"] = df_r["gridPosition"] - df_r["finishPosition"]

# 9) Export
df.reset_index(drop=True, inplace=True)
df_r.reset_index(drop=True, inplace=True)

df.to_csv("fastf1_clean_QR.csv", index=False)
df_r.to_csv("fastf1_race_enriched.csv", index=False)

print("FastF1 clean (Q+R):", df.shape)
print("Race enriched (R + qualyPosition + raceGain):", df_r.shape)

FastF1 clean (Q+R): (4139, 14)
Race enriched (R + qualyPosition + raceGain): (2059, 16)


### 2.1. Test après nettoyage avec assertions

In [ ]:
# TESTS DE VALIDATION FASTF1 CLEANING

assert df["gp"].str.contains("Pre-Season", case=False, na=False).sum() == 0, " Pre-Season encore présent"

assert df.duplicated(subset=["season", "round", "driverCode", "session"]).sum() == 0, "Doublons détectés"

assert (df["maxSpeed_kmh"] == 0).sum() == 0, " maxSpeed 0 encore présent"

for c in ["gridPosition", "finishPosition", "qualyPosition"]:
    assert (df[c] == 0).sum() == 0, f"Valeurs 0 détectées dans {c}"

assert df[(df["session"] == "R") & (df["finishPosition"].isna())].shape[0] == 0, " finishPosition manquant en Race"

assert set(df["session"].unique()) == {"Q", "R"}, " Sessions inattendues"

assert set(df_r["session"].unique()) == {"R"}, "df_r contient autre chose que Race"

print("Tous les tests de validation FastF1 sont passés avec succès.")

## 3. Néttoyage Constructor Finance 

In [3]:


# ==============================
# CONFIG
# ==============================
PROJECT_DIR = Path(r"C:\Users\imane\Desktop\Fil-rouge")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_PATH = RAW_DIR / "f1_constructor_finance_2021_2025.csv"
OUTPUT_PATH = PROCESSED_DIR / "constructor_finance_clean.csv"

# ==============================
# CLEAN FUNCTION
# ==============================
def clean_constructor_finance(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 1) Nettoyage texte
    for c in ["constructorId", "constructorName"]:
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()

    # Harmonisation IDs
    df["constructorId"] = df["constructorId"].str.lower().str.strip()

    # 2) Types
    df["season"] = pd.to_numeric(df["season"], errors="coerce").astype("Int64")
    df["constructorRank"] = pd.to_numeric(df["constructorRank"], errors="coerce").astype("Int64")
    df["constructorPoints"] = pd.to_numeric(df["constructorPoints"], errors="coerce")
    df["costCapUSD"] = pd.to_numeric(df["costCapUSD"], errors="coerce")

    # 3) Valeurs invalides
    df.loc[df["constructorRank"] <= 0, "constructorRank"] = np.nan
    df.loc[df["constructorPoints"] < 0, "constructorPoints"] = np.nan
    df.loc[df["costCapUSD"] <= 0, "costCapUSD"] = np.nan

    # 4) Suppression lignes critiques
    df = df.dropna(subset=["season", "constructorId", "constructorRank", "constructorPoints", "costCapUSD"])

    # 5) Doublons métier
    df = df.drop_duplicates(subset=["season", "constructorId"], keep="first")

    # 6) Reset index
    df.reset_index(drop=True, inplace=True)

    return df

# ==============================
# RUN
# ==============================
df_constructor = pd.read_csv(INPUT_PATH)

df_constructor_clean = clean_constructor_finance(df_constructor)

df_constructor_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("✅ Constructor finance cleaned")
print(" Export :", OUTPUT_PATH)
print("Shape :", df_constructor_clean.shape)

print("\nConstructeurs par saison :")
print(df_constructor_clean.groupby("season")["constructorId"].nunique())

print("\nRésumé numérique :")
print(df_constructor_clean[["constructorRank", "constructorPoints", "costCapUSD"]].describe())

✅ Constructor finance cleaned
 Export : C:\Users\imane\Desktop\Fil-rouge\data\processed\constructor_finance_clean.csv
Shape : (50, 6)

Constructeurs par saison :
season
2021    10
2022    10
2023    10
2024    10
2025    10
Name: constructorId, dtype: int64

Résumé numérique :
       constructorRank  constructorPoints    costCapUSD
count             50.0          50.000000  5.000000e+01
mean               5.5         246.290000  1.380000e+08
std           2.901442         254.065014  4.040610e+06
min                1.0           0.000000  1.350000e+08
25%                3.0          39.250000  1.350000e+08
50%                5.5         128.500000  1.350000e+08
75%                8.0         440.500000  1.400000e+08
max               10.0         860.000000  1.450000e+08


### 3.1. Test après nettoyage 

In [4]:
print("\nVALIDATION CONSTRUCTOR FINANCE\n")

assert df_constructor_clean.duplicated(subset=["season", "constructorId"]).sum() == 0
assert set(df_constructor_clean["season"].unique()) == {2021, 2022, 2023, 2024, 2025}
assert df_constructor_clean["constructorId"].isna().sum() == 0
assert (df_constructor_clean["constructorRank"] <= 0).sum() == 0
assert (df_constructor_clean["constructorPoints"] < 0).sum() == 0
assert (df_constructor_clean["costCapUSD"] <= 0).sum() == 0
assert (df_constructor_clean.groupby("season")["constructorId"].nunique() == 10).all()

print("✅ Constructor finance VALIDATED")


VALIDATION CONSTRUCTOR FINANCE

✅ Constructor finance VALIDATED


## 4. Chargement des datasets nettoyées

In [30]:
df_fastf1 = pd.read_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\processed\fastf1_clean_QR.csv")
df_constructor_fin = pd.read_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\processed\constructor_finance_clean.csv")
df_driver = pd.read_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\processed\driver_finance_clean.csv")

### 4.1. Fusion des datsets 

In [34]:


# ==============================
# STEP 1: DRIVER CODE -> DRIVER ID
# ==============================
driver_map = {
    "VER": "max_verstappen",
    "HAM": "hamilton",
    "BOT": "bottas",
    "PER": "perez",
    "SAI": "sainz",
    "NOR": "norris",
    "LEC": "leclerc",
    "RIC": "ricciardo",
    "GAS": "gasly",
    "ALO": "alonso",
    "OCO": "ocon",
    "VET": "vettel",
    "STR": "stroll",
    "TSU": "tsunoda",
    "RAI": "raikkonen",
    "GIO": "giovinazzi",
    "MSC": "mick_schumacher",
    "MAZ": "mazepin",
    "LAT": "latifi",
    "RUS": "russell",
    "MAG": "kevin_magnussen",
    "ALB": "albon",
    "ZHO": "zhou",
    "DEV": "de_vries",
    "SAR": "sargeant",
    "PIA": "piastri",
    "HUL": "hulkenberg",
    "LAW": "lawson",
    "COL": "colapinto",
    "BEA": "bearman",
    "DOO": "doohan",
    "KUB": "kubica",
    "BOR": "bortoleto",
    "HAD": "hadjar",
    "ANT": "antonelli"
    
}

df_fastf1["driverId"] = df_fastf1["driverCode"].map(driver_map)
print("DriverId manquants après mapping manuel:", df_fastf1["driverId"].isna().sum())

# ==============================
# STEP 1.5: CLEAN DUPLICATES IN DRIVER FINANCE
# ==============================
df_driver = df_driver.copy()

df_driver["_na_count"] = df_driver.isna().sum(axis=1)

sort_cols = ["season", "driverId", "_na_count"]
ascending = [True, True, True]

if "estimatedSalaryUSD" in df_driver.columns:
    sort_cols.append("estimatedSalaryUSD")
    ascending.append(False)

df_driver = df_driver.sort_values(sort_cols, ascending=ascending)

df_driver = df_driver.drop_duplicates(subset=["season", "driverId"], keep="first")
df_driver = df_driver.drop(columns=["_na_count"]).reset_index(drop=True)

print("✅ df_driver après déduplication :", df_driver.shape)
print("Doublons restants :", df_driver.duplicated(subset=["season", "driverId"]).sum())

# ==============================
# STEP 2: MERGE DRIVER FINANCE
# ==============================
df = df_fastf1.merge(
    df_driver,
    on=["season", "driverId"],
    how="left",
    validate="many_to_one"
)

print("Après merge driver finance:", df.shape)
print("DriverId manquants après merge:", df["driverId"].isna().sum())
print("ConstructorId manquants après driver merge:", df["constructorId"].isna().sum())

# ==============================
# STEP 3: COMPLÉTER constructorId AVEC FASTF1
# ==============================
mapping = build_driver_constructor_map()

df = df.merge(
    mapping,
    on=["season", "driverCode"],
    how="left",
    validate="many_to_one"
)

df["constructorId"] = df["constructorId"].fillna(df["constructorId_map"])
df.drop(columns=["constructorId_map"], inplace=True)

print("ConstructorId manquants après complément mapping:", df["constructorId"].isna().sum())

# ==============================
# STEP 4: HARMONISATION CONSTRUCTOR ID
# ==============================
constructor_fix = {
    "astonmartin": "aston_martin",
    "redbull": "red_bull",
    "kick sauber": "sauber",
    "racing bulls": "rb"
}

df["constructorId"] = df["constructorId"].replace(constructor_fix)

print("\nValeurs constructorId dans df mais absentes de df_constructor:")
missing_constructor_ids = sorted(
    set(df["constructorId"].dropna().unique()) - set(df_constructor["constructorId"].dropna().unique())
)
print(missing_constructor_ids)

# ==============================
# STEP 5: MERGE CONSTRUCTOR FINANCE
# ==============================
df = df.merge(
    df_constructor,
    on=["season", "constructorId"],
    how="left",
    validate="many_to_one",
    suffixes=("", "_constructor")
)
# ==============================
# STEP 5: CLEAN FINAL
# ==============================
df = df.drop_duplicates().reset_index(drop=True)




DriverId manquants après mapping manuel: 0
✅ df_driver après déduplication : (110, 12)
Doublons restants : 0
Après merge driver finance: (4139, 25)
DriverId manquants après merge: 0
ConstructorId manquants après driver merge: 0


core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '11'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core        WARNING 	Driver 44 completed the race distance 00:00.067000 before the recorded end of the session.
core           INFO 	Finished loading data for 20 

ConstructorId manquants après complément mapping: 0

Valeurs constructorId dans df mais absentes de df_constructor:
[]


### 4.2. Validation dataset finale 

In [ ]:
# ==============================
# VALIDATION FINALE DATASET
# ==============================

print("Shape:", df.shape)

# 1. Pas de valeurs critiques manquantes
print("\nMissing values:")
print(df[[
    "driverId",
    "constructorId",
    "bestLapTime_sec",
    "avgLapTime_sec"
]].isna().sum())

# 2. Pas de Pre-season
assert not df["gp"].str.contains("Pre-Season|Test", case=False, na=False).any(), \
    "❌ Pre-season détecté"

# 3. Unicité clé métier
duplicates = df.duplicated(subset=["season", "round", "driverCode", "session"]).sum()
print("\nDuplicates:", duplicates)

# 4. Sessions correctes
print("\nSessions:", df["session"].unique())

# 5. Nombre pilotes par course
print("\nDrivers per race:")
print(df.groupby(["season", "round", "session"])["driverCode"].nunique())

print("\n✅ VALIDATION OK")
print(df[["driverId", "constructorId", "constructorRank", "constructorPoints", "costCapUSD"]].isna().sum())


## 5. Exportation du dataset finale 

In [38]:
df.to_csv(r"C:\Users\imane\Desktop\Fil-rouge\data\gold\f1_final_enriched.csv", index=False)